In [ ]:
# -*- coding: utf-8 -*-
"""
Book photo -> EXIF補正 -> page-dewarp (cubic sheet model) -> 出力
- 入出力/一時フォルダはハードコーディング（指示どおり）
- page-dewarp は内部でスパン検出→最適化→リマップまで完結
- デバッグ生成物は ./tmp 配下に保存（削除しない）
依存:
  uv pip install -U page-dewarp opencv-python pillow numpy scipy
"""
from __future__ import annotations
from pathlib import Path
import os, shutil, traceback
from typing import Iterable

import numpy as np
import cv2
from PIL import Image, ImageOps

# page-dewarp (cubic sheet model)
from page_dewarp.image import WarpedImage
from page_dewarp.options.core import Config

# ====== 0) パス設定（ハードコーディング） ======
INPUT_DIR  = Path("./data/sample1/")   # 処理対象画像をここに置く
OUTPUT_DIR = Path("./output")          # 出力先
TMP_DIR    = Path("./tmp")             # 中間結果（削除しない）

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# ====== 1) ユーティリティ ======
def list_images(dirpath: Path) -> Iterable[Path]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    return sorted(p for p in dirpath.glob("*") if p.suffix.lower() in exts)

def save_exif_fixed(src: Path, dst: Path) -> None:
    """EXIFの向きを正した画像を書き出す（OpenCVはEXIFを見ないため）。"""
    with Image.open(src) as im:
        im = ImageOps.exif_transpose(im)  # 端末撮影の回転を補正
        im.save(dst, quality=95)

def autocrop_nonblack(bgr: np.ndarray) -> np.ndarray:
    """黒縁を落とすフォールバック（必要な時のみ使用）"""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > 8
    if not mask.any():
        return bgr
    ys, xs = np.where(mask)
    return bgr[ys.min():ys.max()+1, xs.min():xs.max()+1].copy()

def configure_page_dewarp() -> Config:
    """
    実運用向けの軽いチューニング:
      - NO_BINARY=True : 2値化せずグレースケール出力（OCRや後工程で好都合）
      - SCREEN_MAX_*   : 前処理の縮小サイズ（大きすぎると最適化が重い）
      - PAGE_MARGIN_*  : 外周マスクの余白
      - DEBUG_LEVEL    : 1で要点のみ（対応画像が多いときは0でも可）
    """
    cfg = Config()
    cfg.NO_BINARY      = True     # 2値化OFF（グレースケール保持）  :contentReference[oaicite:2]{index=2}
    cfg.SCREEN_MAX_W   = 1600
    cfg.SCREEN_MAX_H   = 1600
    cfg.PAGE_MARGIN_X  = 12
    cfg.PAGE_MARGIN_Y  = 12
    cfg.OUTPUT_ZOOM    = 1.0      # 出力解像度倍率（>1.0で拡大リサンプリング）
    cfg.REMAP_DECIMATE = 2        # リマップ座標の間引き（偶数が推奨）  :contentReference[oaicite:3]{index=3}
    cfg.DEBUG_LEVEL    = 1        # 0:静か / 1:要所 / 2+:詳細（tmpに可視化が多く出ます）
    return cfg

CFG = configure_page_dewarp()

# ====== 2) メイン処理 ======
def process_one(img_path: Path):
    """
    フロー:
      1) EXIF補正画像を ./tmp に落とす（page-dewarpはOpenCV読み込み）
      2) CWDを ./tmp に一時移動（page-dewarpは相対パスでデバッグ/出力を書く設計）  :contentReference[oaicite:4]{index=4}
      3) WarpedImage(...) 実行 → *_thresh.png が ./tmp に出る（NO_BINARY=True ならグレー）  :contentReference[oaicite:5]{index=5}
      4) 出力を ./output にリネーム・保存、念のため黒縁トリムを適用
    """
    stem = img_path.stem
    exif_fixed = TMP_DIR / f"{stem}.exif.png"
    try:
        save_exif_fixed(img_path, exif_fixed)

        prev_cwd = Path.cwd()
        os.chdir(TMP_DIR)  # デバッグ画像含め tmp に集約
        try:
            print(f"Processing {img_path.name} -> {exif_fixed.name}")
            wi = WarpedImage(str(exif_fixed), config=CFG)  # 内部で最適化→リマップ→保存まで実施
            # 出力ファイル名（RemappedImage 内で "<stem>_thresh.png" を保存）
            produced = TMP_DIR / f"{exif_fixed.stem}_thresh.png"
            print(f"Produced: {produced.name}")
            if not produced.exists():
                # WarpedImage がスパン不足等でスキップした場合
                raise RuntimeError("page-dewarp did not produce output (spans not found?)")

            # 黒縁トリム（必要に応じて）
            print(produced)
            img = cv2.imread(str(produced), cv2.IMREAD_GRAYSCALE)
            print(img)
            if img is None:
                raise RuntimeError("failed to read produced image")
            img = autocrop_nonblack(cv2.cvtColor(img, cv2.COLOR_GRAY2BGR))

            out_path = OUTPUT_DIR / f"{stem}.dewarped.png"
            cv2.imwrite(str(out_path), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
            print(f"[OK] {img_path.name} -> {out_path.name}")
        finally:
            os.chdir(prev_cwd)
    except Exception as e:
        print(f"[ERR] {img_path.name}: {e}")
        traceback.print_exc()

def main():
    imgs = list(list_images(INPUT_DIR))
    if not imgs:
        print(f"Place images in: {INPUT_DIR.as_posix()}")
        return
    for p in imgs:
        print(p)
        process_one(p)
    print(f"Done. tmp kept at: {TMP_DIR.as_posix()}")

if __name__ == "__main__":
    main()


data\sample1\IMG_4152.JPG
Processing IMG_4152.JPG -> IMG_4152.exif.png
[ERR] IMG_4152.JPG: 'NoneType' object has no attribute 'shape'
data\sample1\IMG_4153.JPG
Processing IMG_4152.JPG -> IMG_4152.exif.png
[ERR] IMG_4152.JPG: 'NoneType' object has no attribute 'shape'
data\sample1\IMG_4153.JPG


Traceback (most recent call last):
  File "C:\Users\hiahara\AppData\Local\Temp\ipykernel_11916\4015814393.py", line 90, in process_one
    wi = WarpedImage(str(exif_fixed), config=CFG)  # 内部で最適化→リマップ→保存まで実施
  File "c:\Users\hiahara\Documents\code\python_util\__実装系__\doc_scan\.venv\lib\site-packages\page_dewarp\image.py", line 91, in __init__
    self.small = self.resize_to_screen()
  File "c:\Users\hiahara\Documents\code\python_util\__実装系__\doc_scan\.venv\lib\site-packages\page_dewarp\image.py", line 211, in resize_to_screen
    height, width = self.cv2_img.shape[:2]
AttributeError: 'NoneType' object has no attribute 'shape'


Processing IMG_4153.JPG -> IMG_4153.exif.png
[ERR] IMG_4153.JPG: 'NoneType' object has no attribute 'shape'
data\sample1\IMG_4154.JPG


Traceback (most recent call last):
  File "C:\Users\hiahara\AppData\Local\Temp\ipykernel_11916\4015814393.py", line 90, in process_one
    wi = WarpedImage(str(exif_fixed), config=CFG)  # 内部で最適化→リマップ→保存まで実施
  File "c:\Users\hiahara\Documents\code\python_util\__実装系__\doc_scan\.venv\lib\site-packages\page_dewarp\image.py", line 91, in __init__
    self.small = self.resize_to_screen()
  File "c:\Users\hiahara\Documents\code\python_util\__実装系__\doc_scan\.venv\lib\site-packages\page_dewarp\image.py", line 211, in resize_to_screen
    height, width = self.cv2_img.shape[:2]
AttributeError: 'NoneType' object has no attribute 'shape'
